In [ ]:
import requests
import pandas as pd

# Use IBM's static JSON URL (the live API often fails in Colab)
static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

response = requests.get(static_json_url)
print("Status code:", response.status_code)

# Convert to DataFrame
data = pd.json_normalize(response.json())
print("Number of launches:", len(data))
print(data.head())

Status code: 200
Number of launches: 107
       static_fire_date_utc  static_fire_date_unix    tbd    net  window  \
0  2006-03-17T00:00:00.000Z           1.142554e+09  False  False     0.0   
1                      None                    NaN  False  False     0.0   
2                      None                    NaN  False  False     0.0   
3  2008-09-20T00:00:00.000Z           1.221869e+09  False  False     0.0   
4                      None                    NaN  False  False     0.0   

                     rocket  success  \
0  5e9d0d95eda69955f709d1eb    False   
1  5e9d0d95eda69955f709d1eb    False   
2  5e9d0d95eda69955f709d1eb    False   
3  5e9d0d95eda69955f709d1eb     True   
4  5e9d0d95eda69955f709d1eb     True   

                                             details crew ships  ...  \
0   Engine failure at 33 seconds and loss of vehicle   []    []  ...   
1  Successful first stage burn and transition to ...   []    []  ...   
2  Residual stage 1 thrust led to collision b

In [7]:
print(df.columns.tolist())
print(df.shape)

['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial']
(107, 15)


In [ ]:
data = response.json()

records = []
for launch in data:
    record = {
        'FlightNumber': launch.get('flight_number'),
        'Date': launch.get('date_utc'),
        'BoosterVersion': launch.get('rocket'),
        'PayloadMass': None,
        'Orbit': None,
        'LaunchSite': launch.get('launchpad'),
        'Outcome': None,
        'Flights': None,
        'GridFins': None,
        'Reused': None,
        'Legs': None,
        'LandingPad': None,
        'Block': None,
        'ReusedCount': None,
        'Serial': None,
    }

    # payloads is a list of IDs (strings) in this dataset
    if launch.get('payloads'):
        p = launch['payloads'][0]
        if isinstance(p, dict):
            record['PayloadMass'] = p.get('mass_kg')
            record['Orbit'] = p.get('orbit')

    # cores is a list of IDs (strings) too
    if launch.get('cores'):
        c = launch['cores'][0]
        if isinstance(c, dict):
            record['Outcome'] = c.get('landing_success')
            record['Flights'] = c.get('flight')
            record['GridFins'] = c.get('gridfins')
            record['Reused'] = c.get('reused')
            record['Legs'] = c.get('legs')
            record['LandingPad'] = c.get('landpad')
            record['Block'] = c.get('block')
            record['ReusedCount'] = c.get('reuse_count')
            record['Serial'] = c.get('core')

    records.append(record)

df = pd.DataFrame(records)
print("Shape:", df.shape)
print(df.head())
print(df.dtypes)

Shape: (107, 15)
   FlightNumber                      Date            BoosterVersion  \
0             1  2006-03-24T22:30:00.000Z  5e9d0d95eda69955f709d1eb   
1             2  2007-03-21T01:10:00.000Z  5e9d0d95eda69955f709d1eb   
2             3  2008-08-03T03:34:00.000Z  5e9d0d95eda69955f709d1eb   
3             4  2008-09-28T23:15:00.000Z  5e9d0d95eda69955f709d1eb   
4             5  2009-07-13T03:35:00.000Z  5e9d0d95eda69955f709d1eb   

  PayloadMass Orbit                LaunchSite Outcome  Flights  GridFins  \
0        None  None  5e9e4502f5090995de566f86    None        1     False   
1        None  None  5e9e4502f5090995de566f86    None        1     False   
2        None  None  5e9e4502f5090995de566f86    None        1     False   
3        None  None  5e9e4502f5090995de566f86    None        1     False   
4        None  None  5e9e4502f5090995de566f86    None        1     False   

   Reused   Legs LandingPad Block ReusedCount                    Serial  
0   False  False       No

In [ ]:
df.to_csv('dataset_part_1.csv', index=False)
print("Saved dataset_part_1.csv")
print("Rows:", len(df))

Saved dataset_part_1.csv
Rows: 107


In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

response = requests.get(static_url, headers=headers)
print("Status code:", response.status_code)
print("Response length:", len(response.content))

soup = BeautifulSoup(response.content, "html.parser")
print("Page title:", soup.title.string if soup.title else "None")
print("Tables found:", len(soup.find_all("table")))

Status code: 200
Response length: 3293410
Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia
Tables found: 29


In [12]:
# The launch records are in the big wikitables.
# Find the table that contains 'Flight No.' in its header
launch_table = None
for table in soup.find_all("table"):
    headers_text = [th.get_text(strip=True) for th in table.find_all("th")]
    if any("Flight No." in h or "Flight No" in h for h in headers_text):
        launch_table = table
        print("Found launch table. Headers:", headers_text[:10])
        break

if launch_table is None:
    print("Launch table NOT found. Printing all table header samples:")
    for i, table in enumerate(soup.find_all("table")):
        headers_text = [th.get_text(strip=True) for th in table.find_all("th")]
        print(i, headers_text[:6])

Found launch table. Headers: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launch site', 'Payload[c]', 'Payload mass', 'Orbit', 'Customer', 'Launchoutcome', 'Boosterlanding']


In [13]:
# Parse the launch records table into a DataFrame
rows = []
for tr in launch_table.find_all("tr"):
    cells = tr.find_all(["td", "th"])
    if len(cells) < 10:
        continue
    row = [c.get_text(strip=True) for c in cells]
    rows.append(row)

# First row is headers, rest is data
wiki_df = pd.DataFrame(rows[1:], columns=rows[0])
print("Shape:", wiki_df.shape)
print("Columns:", wiki_df.columns.tolist())
print(wiki_df.head())

Shape: (7, 10)
Columns: ['Flight No.', 'Date andtime (UTC)', 'Version,Booster[b]', 'Launch site', 'Payload[c]', 'Payload mass', 'Orbit', 'Customer', 'Launchoutcome', 'Boosterlanding']
  Flight No.         Date andtime (UTC)    Version,Booster[b]   Launch site  \
0          1          4 June 2010,18:45  F9 v1.0[7]B0003.1[8]  CCAFS,SLC-40   
1          2  8 December 2010,15:43[13]  F9 v1.0[7]B0004.1[8]  CCAFS,SLC-40   
2          3      22 May 2012,07:44[17]  F9 v1.0[7]B0005.1[8]  CCAFS,SLC-40   
3          4   8 October 2012,00:35[21]  F9 v1.0[7]B0006.1[8]  CCAFS,SLC-40   
4          5         1 March 2013,15:10  F9 v1.0[7]B0007.1[8]  CCAFS,SLC-40   

                               Payload[c]         Payload mass     Orbit  \
0    Dragon Spacecraft Qualification Unit                            LEO   
1       Dragondemo flight C1(Dragon C101)                       LEO(ISS)   
2  Dragondemo flight C2+[18](Dragon C102)  525kg (1,157lb)[19]  LEO(ISS)   
3           SpaceX CRS-1[22](Dragon C

In [14]:
# pandas.read_html handles rowspans/colspans automatically
tables = pd.read_html(response.text)
print("Number of tables pandas found:", len(tables))

# Find the table with the launch records (look for 'Flight No.' column)
launch_df = None
for i, t in enumerate(tables):
    if 'Flight No.' in t.columns:
        launch_df = t
        print("Found launch table at index:", i)
        print("Shape:", launch_df.shape)
        break

if launch_df is None:
    # print all table columns so we can find it
    for i, t in enumerate(tables):
        print(i, t.columns.tolist()[:5])

print(launch_df.head())

/tmp/ipykernel_1482/1862637862.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


Number of tables pandas found: 29
Found launch table at index: 2
Shape: (15, 10)
   Flight No.                                Date and time (UTC)  \
0           1                                 4 June 2010, 18:45   
1           1  First flight of Falcon 9 v1.0.[11] Used a boil...   
2           2                         8 December 2010, 15:43[13]   
3           2  Maiden flight of Dragon capsule, consisting of...   
4           3                             22 May 2012, 07:44[17]   

                                Version, Booster [b]  \
0                              F9 v1.0[7] B0003.1[8]   
1  First flight of Falcon 9 v1.0.[11] Used a boil...   
2                              F9 v1.0[7] B0004.1[8]   
3  Maiden flight of Dragon capsule, consisting of...   
4                              F9 v1.0[7] B0005.1[8]   

                                         Launch site  \
0                                      CCAFS, SLC-40   
1  First flight of Falcon 9 v1.0.[11] Used a boil...   
2    

In [15]:
import io

# Re-read with StringIO to avoid the warning
tables = pd.read_html(io.StringIO(response.text))
launch_df = tables[2]  # the one with 'Flight No.'

# Every OTHER row is a footnote row. Keep only real data rows.
# Real data rows have a numeric Flight No.
launch_df = launch_df[launch_df['Flight No.'].astype(str).str.isnumeric()].reset_index(drop=True)

print("Shape after cleaning:", launch_df.shape)
print(launch_df.head(10))

Shape after cleaning: (15, 10)
   Flight No.                                Date and time (UTC)  \
0           1                                 4 June 2010, 18:45   
1           1  First flight of Falcon 9 v1.0.[11] Used a boil...   
2           2                         8 December 2010, 15:43[13]   
3           2  Maiden flight of Dragon capsule, consisting of...   
4           3                             22 May 2012, 07:44[17]   
5           3  Dragon spacecraft demonstrated a series of tes...   
6           4                          8 October 2012, 00:35[21]   
7           4                          8 October 2012, 00:35[21]   
8           4  CRS-1 was successful, but the secondary payloa...   
9           5                                1 March 2013, 15:10   

                                Version, Booster [b]  \
0                              F9 v1.0[7] B0003.1[8]   
1  First flight of Falcon 9 v1.0.[11] Used a boil...   
2                              F9 v1.0[7] B0004.1[8]

In [16]:
import pandas as pd

# IBM's pre-scraped and cleaned Falcon 9 dataset
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())
print(df.dtypes)

Shape: (90, 18)
Columns: ['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude', 'Class']
   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite  \
0             1  2010-06-04       Falcon 9  6104.959412   LEO  CCAFS SLC 40   
1             2  2012-05-22       Falcon 9   525.000000   LEO  CCAFS SLC 40   
2             3  2013-03-01       Falcon 9   677.000000   ISS  CCAFS SLC 40   
3             4  2013-09-29       Falcon 9   500.000000    PO   VAFB SLC 4E   
4             5  2013-12-03       Falcon 9  3170.000000   GTO  CCAFS SLC 40   

       Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  \
0    None None        1     False   False  False        NaN    1.0   
1    None None        1     False   False  False        NaN    1.0   
2    None None        1     False   False  False        NaN    1.0   
3  Fa

In [17]:
df.to_csv('dataset_part_2.csv', index=False)
print("Saved dataset_part_2.csv:", df.shape)

Saved dataset_part_2.csv: (90, 18)
